---
downloads:
  - file: tp05_chute_avec_frottement_teacher.ipynb
    title: Notebook enseignant
    filename: tp05_chute_avec_frottement_enseignant.ipynb
  - file: ../notebooks_student/tp05_chute_avec_frottement_student.ipynb
    title: Notebook étudiant
    filename: tp05_chute_avec_frottement_etudiant.ipynb
---

# TP 05 — Chute avec frottement : résolution numérique

Dans ce TP, nous étudions la chute d'un point matériel soumis à des
frottements de l'air, en résolvant l'équation différentielle du
mouvement de façon analytique puis numérique.

:::{important} Version enseignant
Cette page présente la version **corrigée**, destinée aux enseignants.

Pour télécharger la version à distribuer aux étudiants et à déposer sur
Boostcamp, utilisez le bouton **Notebook étudiant** dans le menu de
téléchargement situé en haut de la page.

Le lien suivant ouvre la version enseignant dans Google Colab :
[Ouvrir dans Google Colab](https://colab.research.google.com/github/ECE-Ecole-Ingenieurs/cours-physique/blob/main/notebooks/tp05_chute_avec_frottement_teacher.ipynb)
:::

**Importation bibliothèques**

In [ ]:
import numpy as np  # Manipulation des vecteurs et opérations matricielles
import matplotlib.pyplot as plt  # Tracé des figures 2D et 3D
from ipywidgets import interact, FloatSlider, IntSlider  # Widgets interactifs (curseurs)
from scipy.integrate import solve_ivp  # Résolution d'EDO par la méthode de Runge-Kutta

## 1. Rappel de cours

Comme au TP précédent, nous partons du principe fondamental de la
dynamique :

$$
\overrightarrow{F} = m\overrightarrow{a}(M)_{/R}
$$

Lorsque les frottements de l'air interviennent, la force de frottement
dépend de la vitesse du point matériel : le PFD projeté sur un axe
conduit alors à une équation différentielle portant sur la vitesse, et
non plus directement à une expression explicite de l'accélération.

Cela revient à résoudre une équation différentielle linéaire du premier
ordre de la forme :

$$
y'(t) + \frac{1}{\tau}y(t) = A
$$

avec $A$ et $\tau$ des constantes. L'ensemble des solutions de cette
équation est de la forme $y=y_p+y_h$, où :

- $y_h$ est solution de l'équation homogène $y'(t)+\frac1\tau y(t)=0$,
  soit $y_h(t)=Ke^{-t/\tau}$ ;
- $y_p$ est une solution particulière constante de l'équation complète,
  soit $y_p=A\tau$.

La solution générale s'écrit donc :

$$
y(t) = A\tau + Ke^{-t/\tau}
$$

où $K$ est déterminée par la condition initiale.

Certaines équations différentielles admettent une solution analytique
exacte, comme dans le cas d'un frottement linéaire. Pour d'autres,
comme le frottement quadratique, la solution n'a pas toujours une
forme simple : on utilise alors une méthode numérique.

Vous avez vu en mathématiques la méthode d'Euler explicite, qui permet
d'approximer la solution d'une équation différentielle du premier
ordre.

On considère une équation sous la forme

$$
y'(t) = F(t, y(t)),
$$

avec, dans notre cas,

$$
F(t,y) = A - \frac{1}{\tau}y.
$$

On subdivise la durée de l'étude en $n$ pas réguliers, c'est-à-dire en
une liste de $n+1$ instants successifs :

$$
t_0,\ t_1,\ t_2,\ \ldots,\ t_i,\ t_{i+1},\ \ldots,\ t_n
$$

où $t_i$ désigne le $i$-ème instant de la liste et $t_{i+1}$ l'instant
suivant. Le pas $h$ est la durée constante entre deux instants
successifs :

$$
h = t_{i+1} - t_i = \frac{t_n - t_0}{n}.
$$

La valeur $y_0=y(t_0)$ à l'instant initial est supposée connue : c'est
la condition initiale.

À partir d'une valeur connue $y_i=y(t_i)$, la méthode d'Euler permet
d'estimer la valeur au pas suivant :

$$
y_{i+1} = y_i + h\,F(t_i, y_i).
$$

On calcule ainsi la solution pas à pas, de $t_0$ jusqu'à $t_n$.

## 2. Chute avec frottement linéaire

On reprend la chute étudiée au TP précédent (axe $(Oz)$ vertical
ascendant, point $M$ de masse $m$ lâché sans vitesse initiale d'une
hauteur $h_0$), en tenant compte cette fois des frottements de l'air.
On modélise la force de frottement par
$\overrightarrow{f}=-\lambda\overrightarrow{v}$, où $\lambda$ est un
coefficient positif.

### 2.1 Théorie

:::{admonition} ✍️ Questions
:class: tip

1. Déterminer l'équation différentielle d'ordre 1 régissant la
   composante $v_z(t)$ de la vitesse du point $M$.
2. En déduire l'expression de $v_z(t)$.
3. En déduire l'expression de l'altitude $z(t)$.
:::

:::{admonition} 💡 Aide
:class: hint

Projetez le PFD sur l'axe $(Oz)$, puis identifiez $\tau$ et $A$ dans
l'équation obtenue pour appliquer la méthode de résolution rappelée
ci-dessus. Pour $z(t)$, intégrez $v_z(t)$ par rapport au temps en
utilisant la condition initiale $z(0)=h_0$.
:::

*Écrivez vos réponses ici (Markdown, $\LaTeX$…) :*

1. --
2. --
3. --

:::{admonition} Réponses
:class: note

1. Le PFD projeté sur $(Oz)$ donne $-mg-\lambda v_z = m\dot v_z$,
   soit :

   $$
   \dot v_z + \frac{\lambda}{m}v_z = -g
   $$
2. C'est une équation de la forme $y'(t)+\frac1\tau y(t)=A$ avec
   $y=v_z$, $\tau=m/\lambda$ et $A=-g$. Avec $v_z(0)=0$, on trouve
   $K=g\tau$, d'où :

   $$
   v_z(t) = g\tau\left(e^{-t/\tau}-1\right)
   $$
3. En intégrant $v_z(t)$ par rapport au temps, avec $z(0)=h_0$ :

   $$
   z(t) = h_0 + g\tau^2\left(1-e^{-t/\tau}\right) - g\tau t
   $$
:::

:::{admonition} Appelez le professeur
:class: warning

Avant de passer à l'implémentation, faites valider votre raisonnement par
l'enseignant.
:::

### 2.2 Implémentation numérique

:::{admonition} 💻 Code
:class: tip

Complétez trois fonctions à partir de vos équations :

1. `vitesse_exacte_frottement_lineaire`, qui renvoie la solution exacte
   $v_z(t)$ ;
2. `position_exacte_frottement_lineaire`, qui renvoie l'altitude
   exacte $z(t)$ ;
3. `vitesse_euler_frottement_lineaire`, qui intègre $v_z(t)$ par la
   méthode d'Euler, à partir de l'équation différentielle établie
   précédemment et de la relation de récurrence
   $v_{i+1}=v_i+hF(t_i,v_i)$.
:::

In [ ]:
def vitesse_exacte_frottement_lineaire(t, m, lam):
    """
    Calcule la vitesse exacte v_z(t) lors d'une chute avec frottement
    linéaire.

    Paramètres
    ----------
    t : float ou np.ndarray
        Instant(s) considéré(s) [s].
    m : float
        Masse du point matériel [kg].
    lam : float
        Coefficient de frottement linéaire [kg/s].

    Renvoie
    -------
    v : float ou np.ndarray
        Vitesse v_z(t) [m/s].
    """
    g = 9.81
    tau = m / lam
    v = g * tau * (np.exp(-t / tau) - 1.0)

    return v

In [ ]:
def position_exacte_frottement_lineaire(t, m, lam, h0):
    """
    Calcule l'altitude exacte z(t) lors d'une chute avec frottement
    linéaire.

    Paramètres
    ----------
    t : float ou np.ndarray
        Instant(s) considéré(s) [s].
    m : float
        Masse du point matériel [kg].
    lam : float
        Coefficient de frottement linéaire [kg/s].
    h0 : float
        Hauteur initiale [m].

    Renvoie
    -------
    z : float ou np.ndarray
        Altitude z(t) [m].
    """
    g = 9.81
    tau = m / lam
    z = h0 + g * tau**2 * (1.0 - np.exp(-t / tau)) - g * tau * t

    return z

In [ ]:
def vitesse_euler_frottement_lineaire(t_vals, m, lam):
    """
    Intègre par la méthode d'Euler explicite la vitesse v_z(t) lors
    d'une chute avec frottement linéaire.

    Paramètres
    ----------
    t_vals : np.ndarray
        Instants successifs, régulièrement espacés [s]. On suppose
        v_z(t_vals[0]) = 0.
    m : float
        Masse du point matériel [kg].
    lam : float
        Coefficient de frottement linéaire [kg/s].

    Renvoie
    -------
    v_vals : np.ndarray
        Vitesse v_z approchée aux instants t_vals [m/s].
    """
    g = 9.81
    h = t_vals[1] - t_vals[0]
    v_vals = np.zeros_like(t_vals)

    for i in range(len(t_vals) - 1):
        v_vals[i + 1] = v_vals[i] + h * (-g - (lam / m) * v_vals[i])

    return v_vals

### 2.3 Analyse

Le code suivant compare la chute avec et sans frottement : à gauche,
un schéma montre les deux masses côte à côte à un instant $t$ donné ;
au centre, l'altitude $z(t)$ des deux chutes ; à droite, la vitesse
exacte, la vitesse approchée par la méthode d'Euler, et la vitesse
sans frottement. Exécutez-le, puis faites varier $\lambda$, la
hauteur initiale $h_0$, l'instant affiché $t$, et le nombre de points
$n_t$ utilisés pour la méthode d'Euler.

In [ ]:
def tracer_vitesse_lineaire(m=1.0, lam=0.5, n_t=100, t=2.0):
    """Compare la chute avec et sans frottement : schéma, altitude et vitesse."""
    g = 9.81
    h0 = 50.0
    t_max = 15.0
    t_affiche = min(t, t_max)

    t_fin = np.linspace(0.0, t_max, 300)

    # Vitesses
    v_exacte = vitesse_exacte_frottement_lineaire(t_fin, m, lam)
    v_sans_frottement = -g * t_fin
    t_euler = np.linspace(0.0, t_max, n_t)
    v_euler = vitesse_euler_frottement_lineaire(t_euler, m, lam)

    # Altitudes (avec frottement : question 3 de la théorie ; sans
    # frottement : chute libre du TP précédent)
    z_avec = np.maximum(position_exacte_frottement_lineaire(t_fin, m, lam, h0), 0.0)
    z_sans = np.maximum(h0 - 0.5 * g * t_fin**2, 0.0)

    # Valeurs à l'instant affiché
    z_avec_actuel = max(position_exacte_frottement_lineaire(t_affiche, m, lam, h0), 0.0)
    z_sans_actuel = max(h0 - 0.5 * g * t_affiche**2, 0.0)
    v_avec_actuel = vitesse_exacte_frottement_lineaire(t_affiche, m, lam)
    v_sans_actuel = -g * t_affiche

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 6), gridspec_kw={'width_ratios': [1, 2, 2]})

    # Schéma : chute avec et sans frottement, côte à côte
    ax1.plot([-0.3, -0.3], [0, h0], color='0.7', linewidth=2)
    ax1.plot([0.3, 0.3], [0, h0], color='0.7', linewidth=2)
    ax1.scatter(-0.3, z_avec_actuel, color='tab:blue', s=120, zorder=5, label='avec frottement')
    ax1.scatter(0.3, z_sans_actuel, color='0.6', s=120, zorder=5, label='sans frottement')
    ax1.axhline(0, color='black', linewidth=1)
    ax1.set_xlim(-1, 1)
    ax1.set_ylim(-2, h0 * 1.1 + 2)
    ax1.set_xticks([])
    ax1.set_ylabel(r'$z$ (m)')
    ax1.set_title(rf'$t={t_affiche:.2f}$ s')
    ax1.legend(loc='upper right', fontsize=8)

    # Altitude z(t)
    ax2.plot(t_fin, z_sans, color='0.6', linestyle='--', linewidth=2, label='sans frottement')
    ax2.plot(t_fin, z_avec, color='tab:blue', linewidth=2, label='avec frottement')
    ax2.scatter(t_affiche, z_sans_actuel, color='0.6', s=60, zorder=5)
    ax2.scatter(t_affiche, z_avec_actuel, color='tab:blue', s=60, zorder=5)
    ax2.set_xlabel(r'$t$ (s)')
    ax2.set_ylabel(r'$z$ (m)')
    ax2.set_title('Altitude en fonction du temps')
    ax2.grid(alpha=0.3)
    ax2.legend(loc='upper right')

    # Vitesse v_z(t)
    ax3.plot(t_fin, v_sans_frottement, color='0.6', linestyle='--', linewidth=2, label='sans frottement')
    ax3.plot(t_fin, v_exacte, color='tab:blue', linewidth=2, label='solution exacte')
    ax3.plot(t_euler, v_euler, color='tab:red', marker='o', markersize=3, linewidth=1, label=rf'Euler ($n_t={n_t}$ points)')
    ax3.scatter(t_affiche, v_sans_actuel, color='0.6', s=60, zorder=5)
    ax3.scatter(t_affiche, v_avec_actuel, color='tab:blue', s=60, zorder=5)
    ax3.set_ylim(-35, 0)
    ax3.set_xlabel(r'$t$ (s)')
    ax3.set_ylabel(r'$v_z$ (m/s)')
    ax3.set_title('Vitesse en fonction du temps')
    ax3.grid(alpha=0.3)
    ax3.legend(loc='upper right')

    fig.tight_layout()
    plt.show()
    plt.close(fig)

tracer_vitesse_lineaire()

In [ ]:
interact(
    tracer_vitesse_lineaire,
    m=FloatSlider(value=1.0, min=1.0, max=3.0, step=0.5, description='Masse (kg)'),
    lam=FloatSlider(value=0.5, min=0.1, max=5.0, step=0.1, description='λ (kg/s)'),
    n_t=IntSlider(value=100, min=3, max=100, step=1, description='Points (n)'),
    t=FloatSlider(value=2.0, min=0.0, max=15.0, step=0.1, description='Temps (s)'),
);

:::{admonition} 🔍 Exploration
:class: tip

1. Faites varier $t$ sur le schéma. Laquelle des deux masses touche le
   sol en premier ? Est-il normal ? Justifiez.
2. Diminuez $\lambda$. Quel est l'effet sur la vitesse de l'objet
   soumis aux forces de frottement ?
3. Quelle est l'allure de la vitesse de l'objet amorti ? Est-ce que
   cette vitesse atteint un plateau ? Comment appelle-t-on cette
   valeur ?
4. Modifiez la masse $m$. A-t-elle une influence sur la vitesse et sur
   la position dans les deux cas ? Justifiez.
5. Comparez la vitesse exacte et la vitesse approchée par la méthode
   d'Euler. Que se passe-t-il lorsque vous diminuez fortement le
   nombre de points $n_t$ ? Essayez de trouver une explication.
:::

*Écrivez vos réponses ici (Markdown, $\LaTeX$…) :*

1. --
2. --
3. --
4. --
5. --

:::{admonition} Réponses
:class: note

1. La masse sans frottement touche le sol en premier : sa vitesse de
   chute croît sans limite, alors que la masse avec frottement est
   freinée par l'air et se rapproche d'une vitesse limite finie. C'est
   normal : le frottement s'oppose au mouvement et ralentit la chute.
2. En diminuant $\lambda$, le frottement diminue : l'objet est de
   moins en moins freiné par l'air, sa vitesse (en valeur absolue)
   augmente plus rapidement et se rapproche de celle de la chute libre
   (à la limite $\lambda\to0$).
3. La vitesse $v_z(t)$ croît rapidement en valeur absolue, puis se
   stabilise : elle atteint un plateau, appelé **vitesse limite**,
   $|v_z|=g\tau=gm/\lambda$.
4. La masse influence la chute avec frottement : $\tau=m/\lambda$
   augmente avec $m$, donc la vitesse limite $|v_z|=g\tau=gm/\lambda$
   augmente aussi — à frottement égal, une masse plus grande est moins
   freinée. Sans frottement en revanche, $v_z(t)=-gt$ ne dépend pas de
   $m$ : la masse n'a aucune influence, comme en chute libre (TP
   précédent).
5. Pour $n_t$ petit (pas $h$ grand), la méthode d'Euler s'écarte
   fortement de la solution exacte, jusqu'à diverger complètement pour
   les valeurs les plus faibles de $n_t$. Pour $n_t$ grand (pas $h$
   petit), les deux courbes deviennent quasiment confondues.
:::

:::{important}
Pour afficher la version interactive, ouvrez le **Notebook enseignant** en
local ou ouvrez ce notebook dans Google Colab.
:::

## 3. Chute avec frottement quadratique

Le modèle de frottement linéaire (§2) n'est valable qu'à faible
vitesse (régime laminaire). Pour une vitesse importante (régime
turbulent), la force de frottement devient proportionnelle au carré
de la vitesse :

$$
\overrightarrow{f} = -\mu\|\overrightarrow{v}\|\overrightarrow{v}
$$

On parle de frottement quadratique, où $\mu$ est une constante qui
dépend du fluide et des caractéristiques de l'objet, de la forme
$\mu=\frac12\rho C_x S$, avec $\rho$ la masse volumique du fluide,
$S$ la surface frontale de l'objet, et $C_x$ le coefficient de
traînée, qui dépend de la **géométrie** du corps (un profil plus
aérodynamique a un $C_x$ plus faible).

### 3.1 Théorie

:::{admonition} ✍️ Questions
:class: tip

1. Déterminer l'équation différentielle d'ordre 1 régissant la
   composante $v_z(t)$ de la vitesse du point $M$.
2. Cette équation possède-t-elle une solution analytique simple, comme
   dans le cas du frottement linéaire ? Justifiez.
3. Sans résoudre complètement l'équation différentielle, déterminer la
   vitesse limite $v_{lim}$ atteinte en régime permanent (lorsque
   $\dot v_z=0$).
:::

:::{admonition} 💡 Aide
:class: hint

Projetez le PFD sur l'axe $(Oz)$. Attention au signe : pendant la
chute, $v_z\leq0$, donc $\|v_z\|=-v_z$. Pour la vitesse limite,
cherchez la valeur constante de $v_z$ qui annule $\dot v_z$.
:::

*Écrivez vos réponses ici (Markdown, $\LaTeX$…) :*

1. --
2. --
3. --

:::{admonition} Réponses
:class: note

1. Le PFD projeté sur $(Oz)$ donne $-mg-\mu\|v_z\|v_z = m\dot v_z$.
   Comme $v_z\leq0$ pendant la chute, $\|v_z\|=-v_z$, donc
   $\|v_z\|v_z=-v_z^2$, d'où :

   $$
   \dot v_z = -g + \frac{\mu}{m}v_z^2
   $$
2. Non : contrairement au frottement linéaire, cette équation est non
   linéaire en $v_z$ (terme en $v_z^2$) ; elle n'admet pas de solution
   analytique simple avec les outils vus jusqu'ici. Il faut la
   résoudre numériquement, par exemple avec la méthode d'Euler.
3. En régime permanent, $\dot v_z=0$, d'où $v_{lim}^2=\dfrac{mg}{\mu}$,
   soit :

   $$
   v_{lim} = -\sqrt{\frac{mg}{\mu}}
   $$

   (négatif, car $M$ chute selon $(Oz)$ descendant).
:::

:::{admonition} Appelez le professeur
:class: warning

Avant de passer à l'implémentation, faites valider votre raisonnement par
l'enseignant.
:::

### 3.2 Implémentation numérique

:::{admonition} 💻 Code
:class: tip

Complétez `vitesse_euler_frottement_quadratique`, qui intègre $v_z(t)$
par la méthode d'Euler, à partir de l'équation différentielle établie
précédemment et de la relation de récurrence
$v_{i+1}=v_i+hF(t_i,v_i)$.
:::

In [ ]:
def vitesse_euler_frottement_quadratique(t_vals, m, mu):
    """
    Intègre par la méthode d'Euler explicite la vitesse v_z(t) lors
    d'une chute avec frottement quadratique.

    Paramètres
    ----------
    t_vals : np.ndarray
        Instants successifs, régulièrement espacés [s]. On suppose
        v_z(t_vals[0]) = 0.
    m : float
        Masse du point matériel [kg].
    mu : float
        Coefficient de frottement quadratique [kg/m].

    Renvoie
    -------
    v_vals : np.ndarray
        Vitesse v_z approchée aux instants t_vals [m/s].
    """
    g = 9.81
    h = t_vals[1] - t_vals[0]
    v_vals = np.zeros_like(t_vals)

    for i in range(len(t_vals) - 1):
        v_vals[i + 1] = v_vals[i] + h * (-g + (mu / m) * v_vals[i]**2)

    return v_vals

### 3.3 Analyse

Le code suivant compare la vitesse obtenue avec frottement quadratique
à celle avec frottement linéaire et sans frottement. Exécutez-le,
puis faites varier $\mu$, $\lambda$, la masse $m$, et le nombre de
points $n_t$.

In [ ]:
def tracer_vitesse_quadratique(m=1.0, lam=0.5, mu=0.1, n_t=100):
    """Compare les vitesses avec frottement quadratique, linéaire et sans frottement."""
    g = 9.81
    t_max = 15.0

    t_euler = np.linspace(0.0, t_max, n_t)
    v_lineaire = vitesse_euler_frottement_lineaire(t_euler, m, lam)
    v_quadratique = vitesse_euler_frottement_quadratique(t_euler, m, mu)
    v_sans_frottement = -g * t_euler

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(t_euler, v_sans_frottement, color='0.6', linestyle='--', linewidth=2, label='sans frottement')
    ax.plot(t_euler, v_lineaire, color='tab:blue', marker='o', markersize=3, linewidth=1, label='frottement linéaire')
    ax.plot(t_euler, v_quadratique, color='tab:red', marker='o', markersize=3, linewidth=1, label='frottement quadratique')
    ax.set_xlabel(r'$t$ (s)')
    ax.set_ylabel(r'$v_z$ (m/s)')
    ax.set_ylim(-100, 0)
    ax.set_title('Frottement linéaire vs quadratique')
    ax.grid(alpha=0.3)
    ax.legend(loc='lower left')

    plt.show()
    plt.close(fig)

tracer_vitesse_quadratique()

In [ ]:
interact(
    tracer_vitesse_quadratique,
    m=FloatSlider(value=1.0, min=1.0, max=3.0, step=0.1, description='Masse (kg)'),
    lam=FloatSlider(value=0.5, min=0.1, max=5.0, step=0.1, description='λ (kg/s)'),
    mu=FloatSlider(value=0.1, min=0.01, max=1.0, step=0.01, description='μ (kg/m)'),
    n_t=IntSlider(value=100, min=3, max=100, step=1, description='Points (n)'),
);

:::{admonition} 🔍 Exploration
:class: tip

1. Comparez les vitesses limites atteintes avec frottement linéaire et
   frottement quadratique.
2. Faites varier $\mu$. Quel est son effet sur la vitesse ?
3. Pour de petites vitesses, les deux modèles de frottement
   (linéaire et quadratique) donnent-ils des résultats proches ? Et
   pour de grandes vitesses ?
4. Réduisez le nombre de points $n_t$. Que constatez-vous ?
:::

*Écrivez vos réponses ici (Markdown, $\LaTeX$…) :*

1. --
2. --
3. --
4. --

:::{admonition} Réponses
:class: note

1. Frottement linéaire : $|v_{lim}|=g\tau=gm/\lambda$ (proportionnelle
   à $m$ et à $1/\lambda$). Frottement quadratique :
   $|v_{lim}|=\sqrt{mg/\mu}$ (proportionnelle à $\sqrt{m}$ et à
   $1/\sqrt{\mu}$) : la dépendance en $m$ et dans le coefficient de
   frottement n'est pas la même.
2. Plus $\mu$ est grand, plus la vitesse limite est petite : le
   frottement quadratique freine d'autant plus la chute.
3. Pour de petites vitesses, les deux modèles donnent des résultats
   proches (le terme quadratique $\mu v_z^2$ reste faible). Pour de
   grandes vitesses, ils s'écartent nettement : le frottement
   quadratique croît beaucoup plus vite avec la vitesse ($v_z^2$
   contre $v_z$), il freine donc davantage la chute aux grandes
   vitesses.
4. Comme pour le frottement linéaire (§2.3), réduire $n_t$ augmente le
   pas $h$ : la méthode d'Euler devient moins précise, jusqu'à
   diverger nettement pour les valeurs les plus faibles de $n_t$.
:::

:::{important}
Pour afficher la version interactive, ouvrez le **Notebook enseignant** en
local ou ouvrez ce notebook dans Google Colab.
:::

## 4. Au-delà d'Euler : `scipy` et méthode de Runge-Kutta

La méthode d'Euler est la plus simple des méthodes numériques de
résolution d'équations différentielles : facile à programmer, mais peu
précise, et instable si le pas de temps est trop grand (très peu de
points $t_i$). Des méthodes plus sophistiquées, comme les méthodes de
**Runge-Kutta**, offrent une bien meilleure précision pour un même pas
de temps.

La bibliothèque `scipy` fournit une implémentation prête à l'emploi de
ces méthodes, via `scipy.integrate.solve_ivp`, qui utilise par défaut
une méthode de Runge-Kutta d'ordre 4(5) (RK45). Le code suivant
compare, pour un grand pas de temps, la solution exacte, la méthode
d'Euler, et Runge-Kutta. Exécutez-le, puis faites varier le nombre de
points $n_t$.

In [ ]:
def vitesse_rk45_frottement_lineaire(t_vals, m, lam):
    """
    Intègre par la méthode de Runge-Kutta (RK45, via scipy) la vitesse
    v_z(t) lors d'une chute avec frottement linéaire. Fonction fournie
    à titre de comparaison : elle ne fait pas partie des questions de
    ce TP.

    Paramètres
    ----------
    t_vals : np.ndarray
        Instants successifs [s]. On suppose v_z(t_vals[0]) = 0.
    m : float
        Masse du point matériel [kg].
    lam : float
        Coefficient de frottement linéaire [kg/s].

    Renvoie
    -------
    v_vals : np.ndarray
        Vitesse v_z approchée aux instants t_vals [m/s].
    """
    g = 9.81

    def F(t, v):
        return -g - (lam / m) * v

    sol = solve_ivp(F, (t_vals[0], t_vals[-1]), [0.0], t_eval=t_vals, method='RK45')

    return sol.y[0]

In [ ]:
def tracer_vitesse_rk45(m=1.0, lam=0.5, n_t=4):
    """Compare la solution exacte, Euler et Runge-Kutta (scipy) pour un grand pas de temps."""
    g = 9.81
    t_max = 15.0

    t_fin = np.linspace(0.0, t_max, 300)
    v_exacte = vitesse_exacte_frottement_lineaire(t_fin, m, lam)

    t_grossier = np.linspace(0.0, t_max, n_t)
    v_euler = vitesse_euler_frottement_lineaire(t_grossier, m, lam)
    v_rk45 = vitesse_rk45_frottement_lineaire(t_grossier, m, lam)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(t_fin, v_exacte, color='0.6', linewidth=2, label='solution exacte')
    ax.plot(t_grossier, v_euler, color='tab:red', marker='o', markersize=5, linewidth=1, label=f'Euler ({n_t} points)')
    ax.plot(t_grossier, v_rk45, color='tab:green', marker='s', markersize=5, linewidth=1, label=f'scipy RK45 ({n_t} points)')
    ax.set_xlabel(r'$t$ (s)')
    ax.set_ylabel(r'$v_z$ (m/s)')
    ax.set_title('Euler vs Runge-Kutta (scipy), grand pas de temps')
    ax.grid(alpha=0.3)
    ax.legend(loc='lower left')

    plt.show()
    plt.close(fig)

tracer_vitesse_rk45()

In [ ]:
interact(
    tracer_vitesse_rk45,
    m=FloatSlider(value=1.0, min=1.0, max=3.0, step=0.1, description='Masse (kg)'),
    lam=FloatSlider(value=0.5, min=0.1, max=5.0, step=0.1, description='λ (kg/s)'),
    n_t=IntSlider(value=4, min=3, max=50, step=1, description='Points (n)'),
);

:::{admonition} 🔍 Exploration
:class: tip

1. Avec les valeurs par défaut ($n_t=4$), la méthode d'Euler diverge
   nettement alors que Runge-Kutta reste proche de la solution exacte.
   Augmentez progressivement $n_t$ : à partir de combien de points la
   méthode d'Euler redevient-elle satisfaisante ?
2. Modifiez $\lambda$. La précision de la méthode d'Euler est-elle
   sensible à ce paramètre, à nombre de points fixé ? Et celle de
   Runge-Kutta ?
:::

*Écrivez vos réponses ici (Markdown, $\LaTeX$…) :*

1. --
2. --

:::{admonition} Réponses
:class: note

1. Pour $\tau=m/\lambda=2$s, l'instabilité d'Euler apparaît lorsque le
   pas $h$ dépasse $2\tau=4$s. Avec $t_{max}=15$s, cela correspond à
   $n_t\lesssim5$ points ; au-delà, la méthode d'Euler redevient
   stable, puis de plus en plus précise à mesure que $n_t$ augmente.
2. Oui : la précision d'Euler dépend directement du rapport
   $h/\tau=h\lambda/m$ (le seuil d'instabilité est $h=2\tau=2m/\lambda$).
   Pour $\lambda$ grand (donc $\tau$ petit), le même pas $h$ devient
   relativement plus grand par rapport à $\tau$, et Euler diverge plus
   facilement. À l'inverse, `solve_ivp` reste précis quelle que soit la
   valeur de $\lambda$ : il ajuste automatiquement son pas interne pour
   respecter une tolérance d'erreur, indépendamment du nombre de points
   demandés en sortie.
:::

:::{important}
Pour afficher la version interactive, ouvrez le **Notebook enseignant** en
local ou ouvrez ce notebook dans Google Colab.
:::

## 5. Application : chute d'un parachutiste « classique »

Un parachutiste saute d'un avion à quelques milliers de mètres
d'altitude. Dans ces conditions, la masse volumique de l'air $\rho$
reste à peu près constante tout au long de la chute libre : c'est
exactement l'hypothèse de notre modèle de frottement quadratique.

### 5.1 Théorie

:::{admonition} ✍️ Questions
:class: tip

1. Parmi les paramètres dont dépend la vitesse limite que vous avez
   déterminée précédemment, lesquels dépendent de l'environnement
   (et échappent donc au contrôle du parachutiste), et lesquels
   dépendent du parachutiste lui-même ?
2. Le coefficient de traînée $C_x$ est-il vraiment une constante ?
   Qu'est-ce qui pourrait le faire varier au cours du saut ?
:::

:::{admonition} 💡 Aide
:class: hint

Identifiez, parmi $g$, $\rho$, $m$, $S$ et $C_x$, lesquels sont
imposés par le lieu et les conditions atmosphériques du saut, et
lesquels sont choisis par le parachutiste (équipement, position du
corps).
:::

*Écrivez vos réponses ici (Markdown, $\LaTeX$…) :*

1. --
2. --

:::{admonition} Réponses
:class: note

1. $g$ et $\rho$ sont imposés par l'environnement (lieu, altitude,
   conditions atmosphériques) : le parachutiste n'a aucune prise
   dessus. En revanche, $m$ (sa masse, équipement compris) et, via sa
   position du corps, $S$ et $C_x$, dépendent du parachutiste
   lui-même : c'est en jouant sur sa masse et sur sa position (bras et
   jambes écartés, tête en bas...) qu'il peut modifier sa vitesse
   limite.
2. Non : $C_x$ dépend de la forme du corps exposé à l'écoulement
   d'air, donc de la position adoptée (position ventrale, tête en
   bas, en boule...) — il n'est constant que tant que le parachutiste
   garde la même position.
:::

### 5.2 Implémentation numérique

En position ventrale classique (bras et jambes écartés, face au sol),
un parachutiste équipé présente typiquement une masse
$m\approx80$ kg, une surface frontale $S\approx0{,}7$ m$^2$, et un
coefficient de traînée $C_x\approx0{,}8$.

:::{admonition} 💻 Code
:class: tip

Complétez la fonction `tracer_parachutiste`, qui trace la vitesse
simulée du parachutiste jusqu'à sa vitesse limite.

- calculez `mu`, le coefficient de frottement quadratique, à partir
  de `rho`, `Cx` et `S` ;
- calculez `v_lim`, la vitesse limite ;
- définissez `t_vals`, un intervalle de temps régulier de 0 à
  `t_max` (utilisez `np.linspace`) ;
- calculez `v_vals`, la vitesse simulée (utilisez
  `vitesse_euler_frottement_quadratique`) ;
- tracez `v_vals` en fonction de `t_vals`, avec une couleur de votre
  choix et une légende ;
- construisez `v_lim_vals`, un tableau de même taille que `t_vals`,
  constant et égal à `v_lim` (utilisez `np.ones_like`), puis tracez-le
  en fonction de `t_vals` ;
- ajoutez un titre et les étiquettes des axes.
:::

In [ ]:
def tracer_parachutiste(m=80.0, S=0.7, Cx=0.8):
    """Simule la vitesse d'un parachutiste jusqu'à la vitesse limite."""
    g = 9.81
    rho = 1.225  # masse volumique de l'air au sol [kg/m^3]  # ne pas modifier
    fig, ax = plt.subplots(figsize=(9, 6))  # ne pas modifier

    # à vous de coder :
    # -----
    mu = 0.5 * rho * Cx * S
    v_lim = np.sqrt(m * g / mu)

    t_max = 20.0
    t_vals = np.linspace(0.0, t_max, 200)
    v_vals = np.abs(vitesse_euler_frottement_quadratique(t_vals, m, mu))
    v_lim_vals = np.ones_like(t_vals) * v_lim

    ax.plot(t_vals, v_vals, color='tab:blue', linewidth=2, label='vitesse simulée')
    ax.plot(t_vals, v_lim_vals, color='tab:red', linestyle='--', linewidth=2, label=f'vitesse limite ({v_lim:.1f} m/s)')
    ax.set_xlabel(r'$t$ (s)')
    ax.set_ylabel(r'$v$ (m/s)')
    ax.set_title('Chute libre d\'un parachutiste')
    # ----- arrêtez-vous ici, il ne reste plus qu'à afficher la figure

    ax.grid(alpha=0.3)
    ax.legend(loc='lower right')

    plt.show()
    plt.close(fig)

tracer_parachutiste()

### 5.3 Analyse

Exécutez le code précédent, puis faites varier $m$, $S$ et $C_x$.

In [ ]:
interact(
    tracer_parachutiste,
    m=FloatSlider(value=80.0, min=50.0, max=120.0, step=5.0, description='Masse (kg)'),
    S=FloatSlider(value=0.7, min=0.2, max=1.0, step=0.05, description='Surface (m²)'),
    Cx=FloatSlider(value=0.8, min=0.3, max=1.2, step=0.05, description='Cx'),
);

:::{admonition} 🔍 Exploration
:class: tip

1. Diminuez $S$ et $C_x$. Quel est l'effet sur la vitesse limite ?
2. Quelle est l'interprétation physique de ces deux paramètres dans
   ce cas ? Qu'est-ce qu'un parachutiste peut faire, concrètement,
   pour augmenter sa vitesse de chute ?
3. Au bout de combien de temps environ la vitesse limite est-elle
   quasiment atteinte ?
:::

*Écrivez vos réponses ici (Markdown, $\LaTeX$…) :*

1. --
2. --
3. --

:::{admonition} Réponses
:class: note

1. En diminuant $S$ et $C_x$, $\mu=\frac12\rho C_xS$ diminue, donc
   $v_{lim}=\sqrt{mg/\mu}$ augmente fortement.
2. $S$ correspond à la surface exposée à l'air (donc à la position du
   corps : bras et jambes écartés ou repliés) et $C_x$ à la forme
   aérodynamique adoptée. En se profilant (bras le long du corps,
   tête en bas, comme en *tracking*), un parachutiste diminue $S$ et
   $C_x$, et peut ainsi atteindre des vitesses bien plus élevées
   (souvent 250 à 300 km/h) qu'en position ventrale classique.
3. La vitesse limite est quasiment atteinte (à 95 % environ) au bout
   d'une dizaine de secondes, ce qui correspond à ce qui est
   généralement observé en chute libre réelle.
:::

:::{important}
Pour afficher la version interactive, ouvrez le **Notebook enseignant** en
local ou ouvrez ce notebook dans Google Colab.
:::

## 6. Application : le saut de Felix Baumgartner (Red Bull Stratos)

Le 14 octobre 2012, le parachutiste autrichien Felix Baumgartner s'est
élancé depuis une capsule à environ 39 km d'altitude (mission *Red
Bull Stratos*), atteignant au passage une vitesse d'environ 377 m/s —
plus rapide que le son à cette altitude. Équipé de son scaphandre, il
présente une masse $m\approx120$ kg, une surface frontale
$S\approx1{,}1$ m$^2$, et un coefficient de traînée $C_x\approx0{,}7$.

Notre modèle de frottement quadratique (§3) suppose une masse
volumique de l'air $\rho$ constante ; il n'est donc valable qu'à basse
altitude, là où l'air est dense et où la vitesse de Baumgartner est
**subsonique**, c'est-à-dire en toute fin de chute libre, juste avant
l'ouverture de son parachute. Comparons la vitesse limite prédite par
notre modèle à la vitesse qu'il a réellement atteinte à ce moment-là.

![Capsule et combinaison de Felix Baumgartner, exposition « Red Bull Stratos » au National Air and Space Museum](https://upload.wikimedia.org/wikipedia/commons/thumb/8/84/RedBullStratosNASM.jpg/330px-RedBullStratosNASM.jpg){width=50%}

*Photo : Eric Long, Smithsonian National Air and Space Museum (domaine public).*

In [ ]:
# Données réelles du saut (Red Bull Stratos, 14/10/2012) :
# temps (s) et vitesse mesurée (km/h)
temps_mesure = np.array([
    0, 2, 5, 8, 10, 12, 15, 17, 19, 21, 23, 25,
    27, 30, 32, 33, 34, 36, 38, 40, 42, 44, 46, 48,
    50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72,
    74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96,
    98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120,
    122, 124, 126, 128, 130, 132, 134, 136, 138, 140, 142, 144,
    146, 148, 150, 152, 154, 156, 158, 160, 162, 164, 166, 168,
    170, 172, 174, 176, 178, 180, 182, 184, 186, 188, 190, 192,
    194, 196, 198, 200, 202, 204, 206, 208, 210, 212, 214, 216,
    218, 220, 222, 224, 226, 228, 230, 232, 234, 236, 238, 240,
    242, 244, 246, 248, 250, 252, 254, 256, 258, 260
])
vitesse_mesure_kmh = np.array([
    0, 67, 187, 287, 364, 430, 530, 612, 670, 749, 808, 866,
    924, 1013, 1076, 1107, 1126, 1174, 1212, 1260, 1285, 1301, 1317, 1340,
    1355, 1334, 1314, 1307, 1261, 1196, 1104, 1056, 993, 957, 917, 850,
    822, 803, 762, 726, 690, 665, 647, 627, 611, 592, 568, 548,
    500, 526, 485, 470, 457, 447, 440, 427, 415, 403, 394, 386,
    375, 372, 374, 375, 375, 373, 371, 367, 361, 359, 358, 361,
    355, 351, 347, 339, 329, 322, 313, 312, 308, 298, 293, 290,
    286, 289, 289, 289, 289, 285, 282, 279, 279, 279, 271, 265,
    259, 254, 241, 229, 221, 223, 220, 224, 221, 220, 222, 221,
    221, 219, 218, 218, 217, 215, 213, 212, 215, 214, 215, 210,
    210, 208, 205, 204, 203, 199, 203, 203, 203, 182
])
vitesse_mesure = vitesse_mesure_kmh / 3.6  # conversion en m/s

In [ ]:
def tracer_baumgartner(rho=1.225):
    """Compare la vitesse mesurée de Baumgartner à la vitesse limite prédite par le modèle quadratique."""
    m = 120.0    # masse équipée [kg]
    S = 1.1      # surface frontale [m^2]
    Cx = 0.7     # coefficient de traînée
    g = 9.81

    mu = 0.5 * rho * Cx * S
    v_lim = np.sqrt(m * g / mu)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(temps_mesure, vitesse_mesure, color='tab:blue', marker='o', markersize=3, linewidth=1, label='vitesse mesurée')
    ax.axhline(v_lim, color='tab:red', linestyle='--', linewidth=2, label=f'vitesse limite modèle ({v_lim:.1f} m/s)')
    ax.set_xlabel(r'$t$ (s)')
    ax.set_ylabel(r'$v$ (m/s)')
    ax.set_title('Chute libre de Felix Baumgartner')
    ax.grid(alpha=0.3)
    ax.legend(loc='upper right')

    plt.show()
    plt.close(fig)

tracer_baumgartner()

In [ ]:
interact(
    tracer_baumgartner,
    rho=FloatSlider(value=1.225, min=0.1, max=1.3, step=0.05, description='ρ air (kg/m³)'),
);

:::{admonition} 🔍 Exploration
:class: tip

1. Comparez la vitesse limite prédite par le modèle
   ($\rho=1{,}225$ kg/m$^3$, air au niveau du sol) à la vitesse
   réellement mesurée en toute fin de chute libre (dernières valeurs
   du graphique). Le modèle est-il satisfaisant dans ce régime ?
2. Diminuez $\rho$ (air moins dense, comme en haute altitude). Que
   devient la vitesse limite prédite par le modèle ? Est-ce cohérent
   avec le fait que Baumgartner ait dépassé la vitesse du son en
   début de chute ?
3. Pourquoi notre modèle ne peut-il pas être utilisé pour décrire
   toute la chute libre de Baumgartner, du saut jusqu'à l'ouverture
   du parachute ?
:::

*Écrivez vos réponses ici (Markdown, $\LaTeX$…) :*

1. --
2. --
3. --

:::{admonition} Réponses
:class: note

1. Le modèle prédit $v_{lim}=\sqrt{mg/\mu}\approx50{,}0$ m/s, très
   proche de la vitesse mesurée en toute fin de chute libre
   ($\approx182$ km/h $\approx50{,}6$ m/s) : un excellent accord, car
   l'air y est effectivement dense, proche de $\rho=1{,}225$ kg/m$^3$.
2. En diminuant $\rho$, $v_{lim}=\sqrt{mg/\mu}$ augmente (puisque
   $\mu\propto\rho$) : un air moins dense freine moins, la vitesse
   limite prédite devient beaucoup plus grande. C'est cohérent : en
   haute altitude, l'air est beaucoup moins dense, ce qui explique
   physiquement pourquoi Baumgartner a pu atteindre une vitesse
   supersonique avant que le frottement ne le ralentisse
   suffisamment.
3. Parce que notre modèle suppose $\rho$ constante, alors que la masse
   volumique de l'air varie énormément entre 39 km d'altitude (air
   extrêmement raréfié) et le niveau du sol (air dense) : il faudrait
   un modèle avec $\rho(z)$ dépendant de l'altitude pour décrire toute
   la chute, ce qui dépasse le cadre de ce TP.
:::

:::{important}
Pour afficher la version interactive, ouvrez le **Notebook enseignant** en
local ou ouvrez ce notebook dans Google Colab.
:::